In [0]:
from pyspark.sql.functions import(col, concat, lit, to_timestamp, to_date,try_to_timestamp, concat_ws,trim, upper, lower )

In [0]:
%run "../includes/common_functions"


In [0]:

%sql
CREATE SCHEMA IF NOT EXISTS f1.silver;

In [0]:
pit_stops_df = spark.read.table("f1.bronze.pit_stops")

In [0]:
pit_stops_renamed_df = pit_stops_df \
    .withColumnRenamed("raceId", "race_id") \
    .withColumnRenamed("driverId", "driver_id") \
    .withColumnRenamed("milliseconds", "pit_stop_milliseconds")

In [0]:
pit_stops_renamed_df = pit_stops_df \
    .withColumn("time", trim(col("time"))) \
    .withColumn("duration", trim(col("duration")))

In [0]:
pit_stops_clean_df = pit_stops_renamed_df \
    .filter(col("race_id").isNotNull()) \
    .filter(col("driver_id").isNotNull()) \
    .dropDuplicates(["race_id", "driver_id", "stop"])

In [0]:
pit_stops_final_df = add_ingestion_date(pit_stops_clean_df) \
    .withColumn("environment", lit("production")) \
    .withColumn("file_date", to_date(lit("2026-05-07")))

In [0]:
pit_stops_final_df.createOrReplaceTempView("pit_stops_updates")

In [0]:
if spark.catalog.tableExists("f1.silver.pit_stops"):

    spark.sql("""

    MERGE INTO f1.silver.pit_stops tgt
    USING pit_stops_updates src
    ON tgt.race_id = src.race_id
    AND tgt.driver_id = src.driver_id
    AND tgt.stop = src.stop

    WHEN MATCHED THEN
    UPDATE SET *

    WHEN NOT MATCHED THEN
    INSERT *

    """)

else:

    pit_stops_final_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("f1.silver.pit_stops")

In [0]:
%sql

SELECT *
FROM f1.silver.pit_stops
LIMIT 10;